# Feature Selection — CatBoost

Optuna-tuned CatBoost is the reference model, then six orthogonal methods prune
the 34 features built in `experiments.ipynb`.

**Pipeline:**
1. Load data + time-based split (last 20% of train as val)
2. Feature engineering (same as experiments.ipynb: time, age, amount, target encodings, velocity)
3. Baseline CatBoost (untuned) — benchmark
4. **Optuna tuning** (50 trials, CatBoost) — finds best params
5. Tuned CatBoost — the reference model for all feature-selection methods
6. Six feature-selection methods (below)
7. Re-fit CatBoost on pruned features
8. Three-way comparison: baseline vs tuned vs pruned

**Feature-selection methods covered:**
1. Correlation analysis — catches redundant features
2. RFE (with CatBoost) — drops features that hurt performance
3. Time consistency — flags features that drift across train windows
4. Adversarial validation — checks train/test similarity (generalization)
5. Permutation importance — true contribution (not just gain-based)
6. CatBoost SHAP — per-prediction interpretability

Final step: combine all 6 to vote on drop candidates, re-fit CatBoost on
the pruned set, and compare full vs pruned.


In [1]:
import pandas as pd
import numpy as np
import catboost as cb
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             fbeta_score, f1_score, precision_score, recall_score)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.feature_selection import RFE
from sklearn.inspection import permutation_importance
from scipy.stats import ks_2samp, spearmanr
import matplotlib.pyplot as plt
import time, gc, warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
pd.set_option('display.max_columns', 100)
%matplotlib inline


In [2]:
TRAIN_PATH = '../data/raw/fraudTrain.csv'
TEST_PATH  = '../data/raw/fraudTest.csv'

train_full = pd.read_csv(TRAIN_PATH).drop(columns=['Unnamed: 0'])
test_full  = pd.read_csv(TEST_PATH ).drop(columns=['Unnamed: 0'])

train_full['trans_date_trans_time'] = pd.to_datetime(train_full['trans_date_trans_time'])
test_full ['trans_date_trans_time'] = pd.to_datetime(test_full ['trans_date_trans_time'])

print(f"Train: {train_full.shape[0]:>9,} rows | fraud: {train_full.is_fraud.sum():>5,} ({train_full.is_fraud.mean():.4%})")
print(f"Test : {test_full.shape[0]:>9,} rows | fraud: {test_full.is_fraud.sum():>5,} ({test_full.is_fraud.mean():.4%})")
print(f"Train window: {train_full.trans_date_trans_time.min()} -> {train_full.trans_date_trans_time.max()}")
print(f"Test  window: {test_full.trans_date_trans_time.min()} -> {test_full.trans_date_trans_time.max()}")


Train: 1,296,675 rows | fraud: 7,506 (0.5789%)
Test :   555,719 rows | fraud: 2,145 (0.3860%)
Train window: 2019-01-01 00:00:18 -> 2020-06-21 12:13:37
Test  window: 2020-06-21 12:14:25 -> 2020-12-31 23:59:34


In [3]:
train_full = train_full.sort_values('trans_date_trans_time').reset_index(drop=True)
cutoff = train_full['trans_date_trans_time'].quantile(0.80)
val_df   = train_full[train_full['trans_date_trans_time'] >= cutoff].reset_index(drop=True)
train_df = train_full[train_full['trans_date_trans_time'] <  cutoff].reset_index(drop=True)

print(f"Train: {len(train_df):>9,} | fraud: {train_df.is_fraud.sum():>4,} ({train_df.is_fraud.mean():.4%}) | {train_df.trans_date_trans_time.min().date()} -> {train_df.trans_date_trans_time.max().date()}")
print(f"Val  : {len(val_df):>9,} | fraud: {val_df.is_fraud.sum():>4,} ({val_df.is_fraud.mean():.4%}) | {val_df.trans_date_trans_time.min().date()} -> {val_df.trans_date_trans_time.max().date()}")
print(f"Test : {len(test_full):>9,} | fraud: {test_full.is_fraud.sum():>4,} ({test_full.is_fraud.mean():.4%}) | {test_full.trans_date_trans_time.min().date()} -> {test_full.trans_date_trans_time.max().date()}")


Train: 1,037,340 | fraud: 5,968 (0.5753%) | 2019-01-01 -> 2020-03-06
Val  :   259,335 | fraud: 1,538 (0.5931%) | 2020-03-06 -> 2020-06-21
Test :   555,719 | fraud: 2,145 (0.3860%) | 2020-06-21 -> 2020-12-31


In [4]:
def build_base_features(df):
    df = df.copy()
    df['hour']     = df['trans_date_trans_time'].dt.hour.astype('int8')
    df['dow']      = df['trans_date_trans_time'].dt.dayofweek.astype('int8')
    df['month']    = df['trans_date_trans_time'].dt.month.astype('int8')
    df['is_night'] = df['hour'].isin([0,1,2,3,4,22,23]).astype('int8')
    df['dob']      = pd.to_datetime(df['dob'])
    df['age']      = ((df['trans_date_trans_time'] - df['dob']).dt.days / 365.25).astype('float32')
    df['amt_log']  = np.log1p(df['amt']).astype('float32')
    df['amt_is_round'] = (df['amt'] == df['amt'].round(0)).astype('int8')
    R = 6371.0
    lat1, lon1 = np.radians(df['lat']),     np.radians(df['long'])
    lat2, lon2 = np.radians(df['merch_lat']), np.radians(df['merch_long'])
    dlat = lat2 - lat1; dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    df['distance_km'] = (2 * R * np.arcsin(np.sqrt(a))).astype('float32')
    return df

train_df = build_base_features(train_df)
val_df   = build_base_features(val_df)
test_df  = build_base_features(test_full)

combined = pd.concat([train_df, val_df, test_df], ignore_index=True)
for col in ['cc_num', 'merchant', 'category', 'city', 'state', 'job', 'zip']:
    counts = combined[col].value_counts().to_dict()
    for d in (train_df, val_df, test_df):
        d[f'{col}_FE'] = d[col].map(counts).astype('float32')
del combined; gc.collect()

GLOBAL = train_df['is_fraud'].mean()
SMOOTHING = 50.0
for col in ['merchant', 'category', 'city', 'state', 'job']:
    stats = train_df.groupby(col)['is_fraud'].agg(['mean', 'count'])
    smoothed = (stats['mean'] * stats['count'] + GLOBAL * SMOOTHING) / (stats['count'] + SMOOTHING)
    smoothed = smoothed.to_dict()
    for d in (train_df, val_df, test_df):
        d[f'{col}_te'] = d[col].map(smoothed).fillna(GLOBAL).astype('float32')

for col in ['merchant', 'category', 'cc_num']:
    stats = train_df.groupby(col)['amt'].agg(['mean', 'std'])
    for stat in ['mean', 'std']:
        colname = f'amt_per_{col}_{stat}'
        mapping = stats[stat].to_dict()
        fallback = train_df['amt'].mean() if stat == 'mean' else train_df['amt'].std()
        for d in (train_df, val_df, test_df):
            d[colname] = d[col].map(mapping).fillna(fallback).astype('float32')

def add_velocity(df, window_hours_list):
    df = df.sort_values(['cc_num', 'trans_date_trans_time']).reset_index(drop=True)
    df['ts'] = df['trans_date_trans_time'].astype('int64') // 10**9
    for h in window_hours_list:
        window_sec = h * 3600
        df[f'txn_last_{h}h'] = (
            df.groupby('cc_num')['ts']
              .transform(lambda s: s.searchsorted(s.values - h*3600, side='right') - 1)
              .astype('float32')
        )
        amt_sums = np.zeros(len(df), dtype='float32')
        for _, g in df.groupby('cc_num', sort=False):
            ts = g['ts'].values; amt = g['amt'].values; idx = g.index.values; n = len(g)
            cum_amt = np.concatenate([[0.0], np.cumsum(amt, dtype='float64')])
            for i in range(n):
                j = np.searchsorted(ts[:i+1], ts[i] - window_sec, side='left')
                amt_sums[idx[i]] = cum_amt[i] - cum_amt[j]
        df[f'amt_sum_last_{h}h'] = amt_sums
    return df.drop(columns=['ts'])

train_df = add_velocity(train_df, [1, 24, 168])
val_df   = add_velocity(val_df,   [1, 24, 168])
test_df  = add_velocity(test_df,  [1, 24, 168])

DROP = ['trans_date_trans_time', 'first', 'last', 'street', 'dob',
        'trans_num', 'unix_time', 'lat', 'long', 'merch_lat', 'merch_long',
        'cc_num', 'merchant', 'category', 'city', 'state', 'job', 'gender', 'zip',
        'is_fraud']
FEATURES = [c for c in train_df.columns if c not in DROP]
print(f"Total features: {len(FEATURES)}")

X_train, y_train = train_df[FEATURES].values, train_df['is_fraud'].values
X_val,   y_val   = val_df  [FEATURES].values, val_df  ['is_fraud'].values
X_test,  y_test  = test_df [FEATURES].values, test_df ['is_fraud'].values
print(f"X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}")


Total features: 34
X_train: (1037340, 34)  X_val: (259335, 34)  X_test: (555719, 34)


In [8]:
# ── Baseline CatBoost on full features ─────────────────────────────────
# This is the reference model. All feature-selection methods below will be
# evaluated by their effect on this model's performance.
# Baseline CatBoost (untuned). This is just a benchmark — the tuned
# CatBoost in cell 7 will replace it as the reference model.
t0 = time.time()
m_cat_base = cb.CatBoostClassifier(
    iterations=500, depth=8, learning_rate=0.05,
    eval_metric='PRAUC', random_seed=42, verbose=0,
    early_stopping_rounds=50, task_type='CPU',
)
m_cat_base.fit(X_train, y_train, eval_set=(X_val, y_val))
t_base = time.time() - t0

val_proba_base  = m_cat_base.predict_proba(X_val) [:, 1]
test_proba_base = m_cat_base.predict_proba(X_test)[:, 1]
base_val_pr  = average_precision_score(y_val,  val_proba_base)
base_test_pr = average_precision_score(y_test, test_proba_base)
base_test_roc = roc_auc_score(y_test, test_proba_base)

print(f"CatBoost BASELINE (untuned, full features) | fit {t_base:.1f}s")
print(f"  val  PR-AUC : {base_val_pr:.4f}")
print(f"  test PR-AUC : {base_test_pr:.4f}")
print(f"  test ROC-AUC: {base_test_roc:.4f}")


CatBoost BASELINE (untuned, full features) | fit 253.5s
  val  PR-AUC : 0.9431
  test PR-AUC : 0.8881
  test ROC-AUC: 0.9952


In [10]:
"""
# ── Optuna tuning for CatBoost (50 trials) ─────────────────────────────
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

def cat_obj(trial):
    params = {
        'iterations':          trial.suggest_int  ('iterations',          300,  1500),
        'depth':               trial.suggest_int  ('depth',               4,    10),
        'learning_rate':       trial.suggest_float('learning_rate',       0.01, 0.15, log=True),
        'l2_leaf_reg':         trial.suggest_float('l2_leaf_reg',         0.5,  20.0, log=True),
        'random_strength':     trial.suggest_float('random_strength',     0.0,  5.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0,  2.0),
        'border_count':        trial.suggest_int  ('border_count',        32,   254),
        'grow_policy': 'SymmetricTree', 'eval_metric': 'PRAUC',
        'random_seed': 42, 'verbose': 0,
        'early_stopping_rounds': 50, 'task_type': 'CPU',
    }
    m = cb.CatBoostClassifier(**params)
    m.fit(X_train, y_train, eval_set=(X_val, y_val))
    return average_precision_score(y_val, m.predict_proba(X_val)[:, 1])

study = optuna.create_study(direction='maximize')
study.optimize(cat_obj, n_trials=50, show_progress_bar=True)

CAT_TUNED = {**study.best_params,
    'grow_policy': 'SymmetricTree', 'eval_metric': 'PRAUC',
    'random_seed': 42, 'verbose': 0,
    'early_stopping_rounds': 50, 'task_type': 'CPU',
}

print(f'Best trial   : #{study.best_trial.number}')
print(f'Best PR-AUC  : {study.best_value:.4f}')
print(f'Best params  : {study.best_params}')
"""

  0%|          | 0/50 [00:00<?, ?it/s]

[W 2026-06-06 09:32:59,746] Trial 29 failed with parameters: {'iterations': 874, 'depth': 5, 'learning_rate': 0.01276734517246222, 'l2_leaf_reg': 5.2957472864849775, 'random_strength': 3.2852224313958374, 'bagging_temperature': 0.9852280853987498, 'border_count': 253} because of the following error: KeyboardInterrupt('').
Traceback (most recent call last):
  File "/home/NullbitZer0/miniconda3/envs/myenv/lib/python3.10/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_881070/765740673.py", line 20, in cat_obj
    m.fit(X_train, y_train, eval_set=(X_val, y_val))
  File "/home/NullbitZer0/miniconda3/envs/myenv/lib/python3.10/site-packages/catboost/core.py", line 5547, in fit
    self._fit(X, y, cat_features, text_features, embedding_features, None, graph, sample_weight, None, None, None, None, baseline, use_best_model,
  File "/home/NullbitZer0/miniconda3/envs/myenv/lib/python3.10/site-packages/catboost/core.py", lin

KeyboardInterrupt: 

In [11]:
# ── Re-train CatBoost with tuned params (this becomes m_cat_full) ──────
# The tuned CatBoost replaces the baseline as the reference model for all
# feature-selection methods below (correlation, RFE, permutation, SHAP).
t0 = time.time()
m_cat_full = cb.CatBoostClassifier(**CAT_TUNED)
m_cat_full.fit(X_train, y_train, eval_set=(X_val, y_val))
t_full = time.time() - t0

val_proba_full  = m_cat_full.predict_proba(X_val) [:, 1]
test_proba_full = m_cat_full.predict_proba(X_test)[:, 1]
full_val_pr  = average_precision_score(y_val,  val_proba_full)
full_test_pr = average_precision_score(y_test, test_proba_full)
full_test_roc = roc_auc_score(y_test, test_proba_full)

print(f'CatBoost TUNED (full features) | fit {t_full:.1f}s')
print(f'  val  PR-AUC : {full_val_pr:.4f}')
print(f'  test PR-AUC : {full_test_pr:.4f}')
print(f'  test ROC-AUC: {full_test_roc:.4f}')


CatBoost TUNED (full features) | fit 271.6s
  val  PR-AUC : 0.9397
  test PR-AUC : 0.8820
  test ROC-AUC: 0.9944


## Feature Selection — Six Methods

Each method produces a list of `drop candidates` for that method.
The final pruning step uses majority voting across methods (drop if
flagged by 2+ methods).


In [13]:
# ── Method 1: Correlation analysis (Spearman) ──────────────────────────
# Spearman captures non-linear monotonic relationships. |r| > 0.95 means one
# feature is essentially a noisy copy of another; drop the simpler one.
print("=" * 70)
print("METHOD 1: CORRELATION (Spearman, threshold=0.95)")
print("=" * 70)

df_train = pd.DataFrame(X_train, columns=FEATURES)
corr = df_train.corr(method='spearman')

CORR_THRESH = 0.95
corr_pairs = []
for i in range(len(FEATURES)):
    for j in range(i+1, len(FEATURES)):
        if abs(corr.iloc[i, j]) > CORR_THRESH:
            corr_pairs.append((FEATURES[i], FEATURES[j], corr.iloc[i, j]))
corr_pairs.sort(key=lambda x: -abs(x[2]))

print(f"\nFound {len(corr_pairs)} pairs with |Spearman r| > {CORR_THRESH}:")
for a, b, r in corr_pairs:
    print(f"  {a:30s} <-> {b:30s}  r={r:+.3f}")

drop_correlation = {a for a, b, r in corr_pairs}
print(f"\nDrop list (Method 1): {sorted(drop_correlation)}")
print(f"  ({len(drop_correlation)} features flagged)")


METHOD 1: CORRELATION (Spearman, threshold=0.95)

Found 6 pairs with |Spearman r| > 0.95:
  amt                            <-> amt_log                         r=+1.000
  txn_last_1h                    <-> txn_last_24h                    r=+1.000
  txn_last_24h                   <-> txn_last_168h                   r=+1.000
  txn_last_1h                    <-> txn_last_168h                   r=+1.000
  amt_per_merchant_mean          <-> amt_per_category_mean           r=+0.969
  cc_num_FE                      <-> zip_FE                          r=+0.964

Drop list (Method 1): ['amt', 'amt_per_merchant_mean', 'cc_num_FE', 'txn_last_1h', 'txn_last_24h']
  (5 features flagged)


In [ ]:
# ── Method 2: Recursive Feature Elimination (with CatBoost) ───────────
# Use a fast CatBoost config (fewer iterations, smaller depth) for RFE speed.
# RFE drops the LEAST important feature (by CatBoost's PredictionValuesChange)
# at each step, retrains, and repeats. The ranking tells you the order in
# which features were dropped (higher = dropped earlier = less important).
print("=" * 70)
print("METHOD 2: RFE (with fast CatBoost, step=2)")
print("=" * 70)

m_rfe = cb.CatBoostClassifier(
    iterations=200, depth=6, learning_rate=0.1,
    eval_metric='PRAUC', random_seed=42, verbose=0,
)
t0 = time.time()
rfe = RFE(estimator=m_rfe, n_features_to_select=15, step=2, verbose=0)
rfe.fit(X_train, y_train)
t_rfe = time.time() - t0

rfe_ranking = pd.Series(rfe.ranking_, index=FEATURES).sort_values()
print(f"\nRFE complete in {t_rfe:.1f}s")
print(f"\nFeatures ranked by RFE (1 = kept, higher = dropped earlier):")
print(rfe_ranking.to_string())

drop_rfe = set(rfe_ranking[rfe_ranking > 1].index)
print(f"\nDrop list (Method 2): {sorted(drop_rfe)}")
print(f"  ({len(drop_rfe)} features flagged)")


METHOD 2: RFE (with fast CatBoost, step=2)

RFE complete in 828.7s

Features ranked by RFE (1 = kept, higher = dropped earlier):
amt                       1
hour                      1
age                       1
is_night                  1
amt_log                   1
cc_num_FE                 1
category_FE               1
merchant_te               1
amt_per_category_mean     1
amt_per_category_std      1
amt_per_cc_num_mean       1
amt_sum_last_24h          1
amt_per_merchant_std      1
city_te                   1
category_te               1
txn_last_1h               2
city_pop                  3
amt_per_cc_num_std        3
merchant_FE               4
amt_sum_last_168h         4
month                     5
job_te                    5
amt_per_merchant_mean     6
dow                       6
zip_FE                    7
amt_sum_last_1h           7
txn_last_24h              8
city_FE                   8
state_te                  9
job_FE                    9
txn_last_168h            10
dis

In [15]:
# ── Method 3: Time consistency (drift across train windows) ────────────
# Split train into 4 chronological windows (rows are time-sorted). For each
# feature, compute mean per window. A feature with >20% max-drift is unstable
# and may be capturing spurious patterns.
print("=" * 70)
print("METHOD 3: TIME CONSISTENCY (4 chronological windows)")
print("=" * 70)

TIME_DRIFT_THRESH = 0.20
n = len(X_train)
windows = np.array_split(np.arange(n), 4)

drift = []
for i, f in enumerate(FEATURES):
    means = [X_train[w, i].mean() for w in windows]
    base = means[0] if abs(means[0]) > 1e-9 else 1e-9
    max_drift = max(abs(m - base) / abs(base) for m in means)
    drift.append((f, max_drift, means))
drift_df = pd.DataFrame(drift, columns=['feature', 'max_drift', 'window_means']) \
              .sort_values('max_drift', ascending=False)

print(f"\nTop 10 features by max drift across time windows:")
for _, row in drift_df.head(10).iterrows():
    print(f"  {row['feature']:30s}  max_drift={row['max_drift']:.3f}")

drop_time = set(drift_df[drift_df.max_drift > TIME_DRIFT_THRESH].feature)
print(f"\nFeatures with >{TIME_DRIFT_THRESH:.0%} drift: {len(drop_time)}/{len(FEATURES)}")
print(f"Drop list (Method 3): {sorted(drop_time)}")


METHOD 3: TIME CONSISTENCY (4 chronological windows)

Top 10 features by max drift across time windows:
  city_pop                        max_drift=0.184
  state_FE                        max_drift=0.133
  amt_sum_last_24h                max_drift=0.067
  city_te                         max_drift=0.064
  amt_sum_last_168h               max_drift=0.056
  job_FE                          max_drift=0.049
  age                             max_drift=0.047
  amt_sum_last_1h                 max_drift=0.043
  amt_per_cc_num_std              max_drift=0.040
  city_FE                         max_drift=0.039

Features with >20% drift: 0/34
Drop list (Method 3): []


In [16]:
# ── Method 4: Adversarial validation (train vs test) ──────────────────
# Train a binary classifier to distinguish train (0) vs test (1). AUC near
# 0.5 = train and test are indistinguishable = good generalization. AUC > 0.7
# = there's significant distribution shift = model may not generalize well.
print("=" * 70)
print("METHOD 4: ADVERSARIAL VALIDATION (CatBoost train vs test)")
print("=" * 70)

X_adv = np.vstack([X_train, X_test])
y_adv = np.concatenate([np.zeros(len(X_train)), np.ones(len(X_test))])
print(f"Adv dataset: {X_adv.shape} | class balance: {y_adv.mean():.4f}")

m_adv = cb.CatBoostClassifier(
    iterations=300, depth=6, learning_rate=0.05,
    eval_metric='AUC', random_seed=42, verbose=0,
)
t0 = time.time()
m_adv.fit(X_adv, y_adv)
t_adv = time.time() - t0

adv_train_proba = m_adv.predict_proba(X_adv)[:, 1]
adv_auc = roc_auc_score(y_adv, adv_train_proba)
adv_imp = pd.Series(m_adv.get_feature_importance(), index=FEATURES) \
             .sort_values(ascending=False)

print(f"\nAdversarial AUC: {adv_auc:.4f}  (fit {t_adv:.1f}s)")
if adv_auc < 0.65:
    print("  -> Good: train and test are hard to distinguish")
elif adv_auc < 0.75:
    print("  -> Mild drift: train and test are partially distinguishable")
else:
    print("  -> WARNING: significant drift between train and test!")

print(f"\nTop 10 features that distinguish train from test:")
print(adv_imp.head(10).to_string())

drop_adv = set(adv_imp.head(5).index)
print(f"\nDrop list (Method 4, top-5 adversarial): {sorted(drop_adv)}")


METHOD 4: ADVERSARIAL VALIDATION (CatBoost train vs test)
Adv dataset: (1593059, 34) | class balance: 0.3488

Adversarial AUC: 1.0000  (fit 68.6s)
  -> WARNING: significant drift between train and test!

Top 10 features that distinguish train from test:
month                  44.319143
txn_last_168h          28.349429
cc_num_FE              19.807251
txn_last_24h            3.234763
txn_last_1h             2.901447
zip_FE                  0.943313
amt_sum_last_168h       0.122759
amt_per_cc_num_mean     0.116861
city_FE                 0.111579
age                     0.058992

Drop list (Method 4, top-5 adversarial): ['cc_num_FE', 'month', 'txn_last_168h', 'txn_last_1h', 'txn_last_24h']


In [17]:
# ── Method 5: Permutation importance (model-agnostic) ─────────────────
# For each feature, shuffle that one column and measure the drop in PR-AUC.
# This is the gold standard for "true" feature importance — it directly
# measures the contribution of each feature to the metric you care about.
print("=" * 70)
print("METHOD 5: PERMUTATION IMPORTANCE (n_repeats=10, on val set)")
print("=" * 70)

t0 = time.time()
r = permutation_importance(
    m_cat_full, X_val, y_val,
    n_repeats=10, scoring='average_precision',
    random_state=42, n_jobs=-1,
)
t_perm = time.time() - t0
perm_imp = pd.Series(r.importances_mean, index=FEATURES) \
              .sort_values(ascending=False)

print(f"\nPermutation importance complete in {t_perm:.1f}s")
print(f"\nTop 15 features by PR-AUC drop when shuffled:")
print(perm_imp.head(15).to_string())
print(f"\nBottom 10 features (least impact when shuffled):")
print(perm_imp.tail(10).to_string())

drop_perm = set(perm_imp.tail(5).index)  # bottom 5
print(f"\nDrop list (Method 5, bottom-5 by permutation): {sorted(drop_perm)}")


METHOD 5: PERMUTATION IMPORTANCE (n_repeats=10, on val set)

Permutation importance complete in 65.1s

Top 15 features by PR-AUC drop when shuffled:
amt                      0.412129
amt_log                  0.259023
hour                     0.208711
category_FE              0.169128
amt_sum_last_24h         0.137518
age                      0.112243
category_te              0.095276
is_night                 0.031526
amt_per_category_std     0.030925
amt_per_cc_num_mean      0.028421
amt_per_category_mean    0.018063
city_pop                 0.016363
amt_per_cc_num_std       0.009413
amt_sum_last_1h          0.009221
amt_per_merchant_std     0.008868

Bottom 10 features (least impact when shuffled):
txn_last_1h     0.000021
amt_is_round    0.000010
job_FE         -0.000138
state_te       -0.000149
distance_km    -0.000153
state_FE       -0.000184
txn_last_24h   -0.000254
month          -0.000295
job_te         -0.004860
city_te        -0.020594

Drop list (Method 5, bottom-5 by permuta

In [19]:
# ── Method 6: CatBoost SHAP values ────────────────────────────────────
import time
print("=" * 70)
print("METHOD 6: CATBOOST SHAP (per-prediction contribution)")
print("=" * 70)

t0 = time.time()

# CatBoost SHAP requires the data to be passed as a Pool
train_pool = cb.Pool(X_train, y_train, feature_names=FEATURES)
shap_vals  = m_cat_full.get_feature_importance(data=train_pool, type='ShapValues')

t_shap = time.time() - t0

# shap_vals shape: (n_samples, n_features + 1). Last column is the expected value.
shap_imp = pd.Series(np.abs(shap_vals[:, :-1]).mean(axis=0), index=FEATURES) \
              .sort_values(ascending=False)

print(f"\nSHAP computed in {t_shap:.1f}s ({shap_vals.shape[0]:,} samples)")
print(f"\nTop 15 features by mean |SHAP|:")
print(shap_imp.head(15).to_string())
print(f"\nBottom 10 features (lowest mean |SHAP|):")
print(shap_imp.tail(10).to_string())

drop_shap = set(shap_imp[shap_imp < shap_imp.median() * 0.01].index)
print(f"\nFeatures with mean |SHAP| < 1% of median: {len(drop_shap)}")
print(f"Drop list (Method 6, near-zero SHAP): {sorted(drop_shap)}")


METHOD 6: CATBOOST SHAP (per-prediction contribution)

SHAP computed in 63.8s (1,037,340 samples)

Top 15 features by mean |SHAP|:
city_te                  0.914809
is_night                 0.738425
amt_sum_last_24h         0.629961
amt                      0.445547
amt_log                  0.408719
hour                     0.406917
amt_per_category_mean    0.371863
category_te              0.369654
age                      0.321453
merchant_te              0.290855
category_FE              0.290576
amt_per_category_std     0.244695
amt_per_cc_num_mean      0.207865
amt_per_merchant_std     0.157492
cc_num_FE                0.139003

Bottom 10 features (lowest mean |SHAP|):
merchant_FE              0.049517
txn_last_1h              0.048910
city_FE                  0.035855
amt_per_merchant_mean    0.028026
job_FE                   0.026986
txn_last_168h            0.021939
distance_km              0.016351
state_FE                 0.013173
state_te                 0.012052
amt_is_roun

In [20]:
# ── Combined ranking: vote across all 6 methods ───────────────────────
# A feature is dropped if it appears in 2+ of the 6 drop lists. Features
# flagged by only 1 method might be quirks of that method (e.g., high
# correlation but high importance = unique signal worth keeping).
print("=" * 70)
print("COMBINED RANKING (majority voting)")
print("=" * 70)

votes = {}
for f in FEATURES:
    votes[f] = sum([
        f in drop_correlation,
        f in drop_rfe,
        f in drop_time,
        f in drop_adv,
        f in drop_perm,
        f in drop_shap,
    ])

vote_df = pd.DataFrame({
    'feature':   FEATURES,
    'votes':     [votes[f] for f in FEATURES],
    'corr':      [f in drop_correlation for f in FEATURES],
    'rfe':       [f in drop_rfe         for f in FEATURES],
    'time':      [f in drop_time        for f in FEATURES],
    'adv':       [f in drop_adv         for f in FEATURES],
    'perm':      [f in drop_perm        for f in FEATURES],
    'shap':      [f in drop_shap        for f in FEATURES],
}).sort_values('votes', ascending=False)

print(f"\nVote distribution:")
for v in range(7):
    n = (vote_df['votes'] == v).sum()
    if n > 0:
        print(f"  {v} votes: {n} features")

DROPPED = vote_df[vote_df['votes'] >= 2].feature.tolist()
KEPT    = [f for f in FEATURES if f not in DROPPED]
print(f"\nFinal drop list ({len(DROPPED)} features, 2+ votes): {sorted(DROPPED)}")
print(f"\nFinal keep list ({len(KEPT)} features):")
for f in KEPT:
    print(f"  {f}")


COMBINED RANKING (majority voting)

Vote distribution:
  0 votes: 12 features
  1 votes: 13 features
  2 votes: 6 features
  3 votes: 2 features
  4 votes: 1 features

Final drop list (9 features, 2+ votes): ['amt_is_round', 'amt_per_merchant_mean', 'cc_num_FE', 'job_te', 'month', 'state_FE', 'txn_last_168h', 'txn_last_1h', 'txn_last_24h']

Final keep list (25 features):
  amt
  city_pop
  hour
  dow
  is_night
  age
  amt_log
  distance_km
  merchant_FE
  category_FE
  city_FE
  job_FE
  zip_FE
  merchant_te
  category_te
  city_te
  state_te
  amt_per_merchant_std
  amt_per_category_mean
  amt_per_category_std
  amt_per_cc_num_mean
  amt_per_cc_num_std
  amt_sum_last_1h
  amt_sum_last_24h
  amt_sum_last_168h


In [ ]:
## drop 3

DROPPED = vote_df[vote_df['votes'] >= 3].feature.tolist()
KEPT    = [f for f in FEATURES if f not in DROPPED]
print(f"\nFinal drop list ({len(DROPPED)} features, 3+ votes): {sorted(DROPPED)}")
print(f"\nFinal keep list ({len(KEPT)} features):")
for f in KEPT:
    print(f"  {f}")
    


Final drop list (3 features, 2+ votes): ['month', 'txn_last_1h', 'txn_last_24h']

Final keep list (31 features):
  amt
  city_pop
  hour
  dow
  is_night
  age
  amt_log
  amt_is_round
  distance_km
  cc_num_FE
  merchant_FE
  category_FE
  city_FE
  state_FE
  job_FE
  zip_FE
  merchant_te
  category_te
  city_te
  state_te
  job_te
  amt_per_merchant_mean
  amt_per_merchant_std
  amt_per_category_mean
  amt_per_category_std
  amt_per_cc_num_mean
  amt_per_cc_num_std
  amt_sum_last_1h
  amt_sum_last_24h
  txn_last_168h
  amt_sum_last_168h


In [27]:
## drop 4 
DROPPED = vote_df[vote_df['votes'] >= 4].feature.tolist()
KEPT    = [f for f in FEATURES if f not in DROPPED]
print(f"\nFinal drop list ({len(DROPPED)} features, 4 votes): {sorted(DROPPED)}")
print(f"\nFinal keep list ({len(KEPT)} features):")
for f in KEPT:
    print(f"  {f}")


Final drop list (1 features, 4 votes): ['txn_last_24h']

Final keep list (33 features):
  amt
  city_pop
  hour
  dow
  month
  is_night
  age
  amt_log
  amt_is_round
  distance_km
  cc_num_FE
  merchant_FE
  category_FE
  city_FE
  state_FE
  job_FE
  zip_FE
  merchant_te
  category_te
  city_te
  state_te
  job_te
  amt_per_merchant_mean
  amt_per_merchant_std
  amt_per_category_mean
  amt_per_category_std
  amt_per_cc_num_mean
  amt_per_cc_num_std
  txn_last_1h
  amt_sum_last_1h
  amt_sum_last_24h
  txn_last_168h
  amt_sum_last_168h


In [28]:
# ── Re-fit CatBoost on pruned feature set ──────────────────────────────
t0 = time.time()
m_cat_pruned = cb.CatBoostClassifier(
    iterations=500, depth=8, learning_rate=0.05,
    eval_metric='PRAUC', random_seed=42, verbose=0,
    early_stopping_rounds=50, task_type='CPU',
)

X_train_p = pd.DataFrame(X_train, columns=FEATURES)[KEPT].values
X_val_p   = pd.DataFrame(X_val,   columns=FEATURES)[KEPT].values
X_test_p  = pd.DataFrame(X_test,  columns=FEATURES)[KEPT].values

m_cat_pruned.fit(X_train_p, y_train, eval_set=(X_val_p, y_val))
t_pruned = time.time() - t0

val_proba_pruned  = m_cat_pruned.predict_proba(X_val_p) [:, 1]
test_proba_pruned = m_cat_pruned.predict_proba(X_test_p)[:, 1]
pruned_val_pr  = average_precision_score(y_val,  val_proba_pruned)
pruned_test_pr = average_precision_score(y_test, test_proba_pruned)
pruned_test_roc = roc_auc_score(y_test, test_proba_pruned)

print(f"CatBoost (pruned: {len(KEPT)} features) | fit {t_pruned:.1f}s")
print(f"  val  PR-AUC : {pruned_val_pr:.4f}")
print(f"  test PR-AUC : {pruned_test_pr:.4f}")
print(f"  test ROC-AUC: {pruned_test_roc:.4f}")


CatBoost (pruned: 33 features) | fit 208.5s
  val  PR-AUC : 0.9360
  test PR-AUC : 0.8776
  test ROC-AUC: 0.9946


In [29]:
# ── Comparison: full vs pruned CatBoost ───────────────────────────────
print("=" * 70)
print("FULL vs PRUNED CATBOOST")
print("=" * 70)
print(f"{'Metric':<22s} {'Baseline':>10s} {'Tuned':>10s} {'Pruned':>10s}")
print("-" * 70)
print(f"{'# features':<22s} {len(FEATURES):>10d} {len(FEATURES):>10d} {len(KEPT):>10d}")
print(f"{'Fit time (s)':<22s} {t_base:>10.1f} {t_full:>10.1f} {t_pruned:>10.1f}")
print(f"{'val  PR-AUC':<22s} {base_val_pr:>10.4f} {full_val_pr:>10.4f} {pruned_val_pr:>10.4f}")
print(f"{'test PR-AUC':<22s} {base_test_pr:>10.4f} {full_test_pr:>10.4f} {pruned_test_pr:>10.4f}")
print(f"{'test ROC-AUC':<22s} {base_test_roc:>10.4f} {full_test_roc:>10.4f} {pruned_test_roc:>10.4f}")

print(f"\nDeltas (tuned - base, pruned - tuned):")
print(f"  test PR-AUC:  {full_test_pr - base_test_pr:+.4f} (tuning), {pruned_test_pr - full_test_pr:+.4f} (pruning)")
print(f"  test ROC-AUC: {full_test_roc - base_test_roc:+.4f} (tuning), {pruned_test_roc - full_test_roc:+.4f} (pruning)")

print(f"\nVerdict: ", end='')
delta_pr = pruned_test_pr - full_test_pr  # pruned vs tuned
if delta_pr >= -0.001:
    print(f"PRUNING IS A WIN (delta = {delta_pr:+.4f}). Use the pruned model.")
    print(f"  Same accuracy with {len(FEATURES)-len(KEPT)} fewer features and {100*(1-t_pruned/t_full):.0f}% faster training.")
elif delta_pr >= -0.005:
    print(f"PRUNING IS ACCEPTABLE (delta = {delta_pr:+.4f}). Marginal accuracy loss for cleaner model.")
else:
    print(f"PRUNING HURTS (delta = {delta_pr:+.4f}). Stick with full features.")


FULL vs PRUNED CATBOOST
Metric                   Baseline      Tuned     Pruned
----------------------------------------------------------------------
# features                     34         34         33
Fit time (s)                253.5      271.6      208.5
val  PR-AUC                0.9431     0.9397     0.9360
test PR-AUC                0.8881     0.8820     0.8776
test ROC-AUC               0.9952     0.9944     0.9946

Deltas (tuned - base, pruned - tuned):
  test PR-AUC:  -0.0060 (tuning), -0.0044 (pruning)
  test ROC-AUC: -0.0008 (tuning), +0.0002 (pruning)

Verdict: PRUNING IS ACCEPTABLE (delta = -0.0044). Marginal accuracy loss for cleaner model.


In [23]:
# ── 3-tier analysis on the chosen model ────────────────────────────────
# Use the pruned model (or full, depending on verdict above) for the final
# production-style 3-tier threshold analysis.
USE_PRUNED = (pruned_test_pr >= full_test_pr - 0.001)
val_proba_final  = val_proba_pruned  if USE_PRUNED else val_proba_full
test_proba_final = test_proba_pruned if USE_PRUNED else test_proba_full
final_name = "pruned" if USE_PRUNED else "full"
final_n_features = len(KEPT) if USE_PRUNED else len(FEATURES)

thresholds = np.linspace(0.001, 0.99, 1000)
f2_scores  = [fbeta_score(y_val, (val_proba_final >= t).astype(int), beta=2, zero_division=0)
              for t in thresholds]
best_t = thresholds[int(np.argmax(f2_scores))]
print(f"Best model: {final_name} ({final_n_features} features)")
print(f"F2-optimal threshold on val: {best_t:.4f}")

precisions = np.array([precision_score(y_test, (test_proba_final >= t).astype(int), zero_division=0)
                       for t in thresholds])
recalls    = np.array([recall_score   (y_test, (test_proba_final >= t).astype(int), zero_division=0)
                       for t in thresholds])
mask_p = precisions >= 0.95
mask_r = recalls >= 0.95
t_tier1 = float(thresholds[np.argmax(mask_p)]) if mask_p.any() else None
t_tier3 = float(thresholds[len(thresholds) - 1 - np.argmax(mask_r[::-1])]) if mask_r.any() else None
t_tier2 = best_t

print(f"\n{'Tier':<26s} {'Threshold':>10s} {'Precision':>10s} {'Recall':>8s} {'F1':>8s} {'TP':>6s} {'FP':>6s}")
print("-" * 80)
for label, t in [('1: Auto-block (high conf)', t_tier1),
                 ('2: Review queue (F2)',      t_tier2),
                 ('3: Soft signal (high rec)', t_tier3)]:
    if t is None:
        print(f"{label:<26s}  (no threshold meets target)")
        continue
    preds = (test_proba_final >= t).astype(int)
    p = precision_score(y_test, preds, zero_division=0)
    r = recall_score   (y_test, preds, zero_division=0)
    f = f1_score       (y_test, preds, zero_division=0)
    tp = int(((preds == 1) & (y_test == 1)).sum())
    fp = int(((preds == 1) & (y_test == 0)).sum())
    print(f"{label:<26s} {t:>10.4f} {p:>10.4f} {r:>8.4f} {f:>8.4f} {tp:>6d} {fp:>6d}")


Best model: full (34 features)
F2-optimal threshold on val: 0.0950

Tier                        Threshold  Precision   Recall       F1     TP     FP
--------------------------------------------------------------------------------
1: Auto-block (high conf)      0.5653     0.9504   0.6960   0.8036   1493     78
2: Review queue (F2)           0.0950     0.7179   0.8564   0.7810   1837    722
3: Soft signal (high rec)      0.0050     0.2041   0.9543   0.3363   2047   7982


## Summary

**Pipeline:**
- Baseline CatBoost → Optuna tuning (50 trials) → Tuned CatBoost
- 6 feature-selection methods applied to tuned CatBoost
- Re-fit on pruned features; 3-way comparison (baseline / tuned / pruned)

**Feature selection results:**

- 6 orthogonal methods were applied to the 34 engineered features
- Drop rule: feature flagged by 2+ methods (majority vote)
- Adversarial validation AUC: see Method 4 output (low = good generalization)
- Final feature count: see Comparison cell

**Operational story:**

- The 3-tier analysis (cell 16) shows the same production-ready story as
  `experiments.ipynb`, with potentially fewer features and faster training
- The choice between `full` and `pruned` CatBoost is dictated by the
  verdict in the Comparison cell

**Next steps (out of scope):**

- Stability selection (bootstrap subsets, see which features are
  consistently selected)
- Per-segment feature analysis (which features matter for online vs
  in-person transactions)
- Time-aware feature selection (separate feature sets for early vs
  late fraud patterns)


removing 4 falgged feture is ok
